In [1]:
from transformers import AutoModel, AutoModelForCausalLM, AutoConfig

from transformers.models.qwen3.modeling_qwen3 import Qwen3Model

from hybrid_qwen import HyperbolicQwenConfig

import torch

/nfs/roberts/project/pi_zy286/bml62/hyperbolic_rag/vllm_rag_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Registering HyperbolicQwen


In [2]:
model = AutoModel.from_pretrained("Qwen/Qwen3-1.7B")
model2 = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5282.48it/s]
Qwen3Model LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 52401.42it/s]


In [3]:
print(model)

Qwen3Model(
  (embed_tokens): Embedding(151936, 2048)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
        (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
        (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [4]:
config = HyperbolicQwenConfig()

print(config.hidden_size)

base_config = AutoConfig.from_pretrained("Qwen/Qwen3-1.7B")

new_config = HyperbolicQwenConfig(**base_config.to_dict(), k=1.0)

# print(config)

print(base_config.hidden_size)
print(new_config.hidden_size)

4096
2048
2048


In [5]:
print(model2.lm_head)

print(f"Weights shape: {model2.lm_head.weight.shape}")

a = torch.rand(size=(2, 2048), dtype=torch.bfloat16)
b = model2.lm_head(a)

print(f"Shape: {b.shape}")

print(b[0][512])

weights = model2.lm_head.weight

print(weights.shape)

t = torch.matmul(a, weights.T)

print(t.shape)

print(t[0, 512])

Linear(in_features=2048, out_features=151936, bias=False)
Weights shape: torch.Size([151936, 2048])
Shape: torch.Size([2, 151936])
tensor(1.4219, dtype=torch.bfloat16, grad_fn=<SelectBackward0>)
torch.Size([151936, 2048])
torch.Size([2, 151936])
tensor(1.4219, dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


In [16]:
print(weights.shape)

n = weights.norm(dim=1, keepdim=False)

print(n.shape)
print(n)

torch.Size([151936, 2048])
torch.Size([151936])
tensor([1.5703, 1.6484, 1.5156,  ..., 0.3477, 0.3477, 0.3477],
       dtype=torch.bfloat16, grad_fn=<LinalgVectorNormBackward0>)


In [ ]:
from hybrid_qwen import HyperbolicProjection

proj = HyperbolicProjection(k=1.0)

w_hyp = proj(weights)

print(weights)
print(w_hyp)

Parameter containing:
tensor([[-0.0127,  0.0195,  0.0117,  ...,  0.0157, -0.0469, -0.0013],
        [ 0.0249, -0.0055, -0.0674,  ...,  0.0037,  0.0403, -0.0165],
        [-0.0173, -0.0284, -0.0530,  ..., -0.0305, -0.0074,  0.0649],
        ...,
        [ 0.0060,  0.0131,  0.0190,  ...,  0.0020, -0.0014, -0.0055],
        [ 0.0060,  0.0131,  0.0190,  ...,  0.0020, -0.0014, -0.0055],
        [ 0.0060,  0.0131,  0.0190,  ...,  0.0020, -0.0014, -0.0055]],
       dtype=torch.bfloat16, requires_grad=True)
tensor([[ 2.5000e+00, -1.8555e-02,  2.8442e-02,  ...,  2.2949e-02,
         -6.8359e-02, -1.9302e-03],
        [ 2.6875e+00,  3.7598e-02, -8.3618e-03,  ...,  5.5847e-03,
          6.1035e-02, -2.5024e-02],
        [ 2.3906e+00, -2.4780e-02, -4.0771e-02,  ..., -4.3701e-02,
         -1.0559e-02,  9.2773e-02],
        ...,
        [ 1.0625e+00,  6.1951e-03,  1.3428e-02,  ...,  2.0752e-03,
         -1.4572e-03, -5.6152e-03],
        [ 1.0625e+00,  6.1951e-03,  1.3428e-02,  ...,  2.0752e-03,
   

In [6]:
print(Qwen3Model)

<class 'transformers.models.qwen3.modeling_qwen3.Qwen3Model'>


In [7]:
# Try importing the Lorentz model and do a quick sanity check
from geoopt.manifolds import Lorentz
import torch

k = 1.0

# Note that k is the negative curvature per the docs
hyperboloid = Lorentz(k=k)

u = torch.tensor([[1., 0., 0., 0.], [0., 1., 0., 0.]])
z = torch.zeros(size=(u.shape[0], 1))

u = torch.cat((z, u), dim=1)

p = torch.zeros(size=u.shape)
p[:, 0] = 1/(k ** 0.5)

print(u)
print(p)

# Exp_x(u)
u_hyp = hyperboloid.expmap(p, u)

print(u_hyp)

print(hyperboloid.k)

tensor([[0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.]])
tensor([[1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0.]])
tensor([[1.5431, 1.1752, 0.0000, 0.0000, 0.0000],
        [1.5431, 0.0000, 1.1752, 0.0000, 0.0000]])
Parameter containing:
tensor(1.)


In [ ]:
# Try importing the Lorentz model and do a quick sanity check
from geoopt.manifolds import Lorentz
import torch

k = 1.0

# Note that k is the negative curvature per the docs
hyperboloid = Lorentz(k=k)

# Assume u of form [batch, length, emb_dim]

u = torch.tensor([[[1., 0.]], [[0., 1.]], [[-1., 0.]]])

*batch_dims, d = u.shape

z = torch.zeros(size=(*batch_dims, 1))

u = torch.cat((z, u), dim=2)

p = torch.zeros(size=u.shape)
p[:, :, 0] = 1/(k ** 0.5)

print(u)
print(p)

# Exp_x(u)
u_hyp = hyperboloid.expmap(p, u)

print(u_hyp)

print(hyperboloid.k) 

tensor([[[ 0.,  1.,  0.]],

        [[ 0.,  0.,  1.]],

        [[ 0., -1.,  0.]]])
tensor([[[1., 0., 0.]],

        [[1., 0., 0.]],

        [[1., 0., 0.]]])
tensor([[[ 1.5431,  1.1752,  0.0000]],

        [[ 1.5431,  0.0000,  1.1752]],

        [[ 1.5431, -1.1752,  0.0000]]])
Parameter containing:
tensor(1.)


In [9]:
a = torch.tensor([1])
*batch_dims, d = a.shape

print(batch_dims)
print(d)

b = torch.zeros(size=(*batch_dims, d))
print(b)

[]
1
tensor([0.])


In [10]:
*batch_dims, d = u.shape

print(batch_dims)
print(d)

[3, 1]
3


In [11]:
from transformers.models.qwen3.configuration_qwen3 import Qwen3Config

c = Qwen3Config()

print(c)

Qwen3Config {
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "eos_token_id": null,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 22016,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32

In [12]:
from transformers import AutoModelForCausalLM
print(type(AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")))

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 53357.41it/s]


<class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
